In [1]:
import spacy
import numpy as np
import pandas as pd
from spacy.training import offsets_to_biluo_tags
nlp = spacy.load("en_core_web_lg")
from tqdm import trange
import torch
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from pytorch_pretrained_bert import BertTokenizer, BertConfig
from pytorch_pretrained_bert import BertForTokenClassification, BertAdam
from seqeval.metrics import classification_report, accuracy_score, f1_score

In [2]:
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")
doc
for token in doc:
    print(token.text, token.pos_, token.dep_)

Apple PROPN nsubj
is AUX aux
looking VERB ROOT
at ADP prep
buying VERB pcomp
U.K. PROPN nsubj
startup VERB ccomp
for ADP prep
$ SYM quantmod
1 NUM compound
billion NUM pobj


In [3]:
# Adding '\n' to the default spacy tokenizer
prefixes = ('\n', ) + tuple(nlp.Defaults.prefixes)
prefix_regex = spacy.util.compile_prefix_regex(prefixes)
nlp.tokenizer.prefix_search = prefix_regex.search

In [4]:
# Personal Custom Tags Dictionary
entity_dict = {
    'Name': 'NAME', 
    'College Name': 'CLG',
    'Degree': 'DEG',
    'Graduation Year': 'GRADYEAR',
    'Years of Experience': 'YOE',
    'Companies worked at': 'COMPANY',
    'Designation': 'DESIG',
    'Skills': 'SKILLS',
    'Location': 'LOC',
    'Email Address': 'EMAIL'
}

In [5]:
import os
print(os.path.exists('./data.json'))

True


In [2]:
with open('./data.json', 'r', encoding='utf-8') as f:
    for i in range(1):
        print(f.readline())

{"content": "Afreen Jamadar\nActive member of IIIT Committee in Third year\n\nSangli, Maharashtra - Email me on Indeed: indeed.com/r/Afreen-Jamadar/8baf379b705e37c6\n\nI wish to use my knowledge, skills and conceptual understanding to create excellent team\nenvironments and work consistently achieving organization objectives believes in taking initiative\nand work to excellence in my work.\n\nWORK EXPERIENCE\n\nActive member of IIIT Committee in Third year\n\nCisco Networking -  Kanpur, Uttar Pradesh\n\norganized by Techkriti IIT Kanpur and Azure Skynet.\nPERSONALLITY TRAITS:\n• Quick learning ability\n• hard working\n\nEDUCATION\n\nPG-DAC\n\nCDAC ACTS\n\n2017\n\nBachelor of Engg in Information Technology\n\nShivaji University Kolhapur -  Kolhapur, Maharashtra\n\n2016\n\nSKILLS\n\nDatabase (Less than 1 year), HTML (Less than 1 year), Linux. (Less than 1 year), MICROSOFT\nACCESS (Less than 1 year), MICROSOFT WINDOWS (Less than 1 year)\n\nADDITIONAL INFORMATION\n\nTECHNICAL SKILLS:\n\n• 

In [4]:
with open('./data.json', 'r', encoding='utf-8') as f:
    print(f.readline())

{"content": "Afreen Jamadar\nActive member of IIIT Committee in Third year\n\nSangli, Maharashtra - Email me on Indeed: indeed.com/r/Afreen-Jamadar/8baf379b705e37c6\n\nI wish to use my knowledge, skills and conceptual understanding to create excellent team\nenvironments and work consistently achieving organization objectives believes in taking initiative\nand work to excellence in my work.\n\nWORK EXPERIENCE\n\nActive member of IIIT Committee in Third year\n\nCisco Networking -  Kanpur, Uttar Pradesh\n\norganized by Techkriti IIT Kanpur and Azure Skynet.\nPERSONALLITY TRAITS:\n• Quick learning ability\n• hard working\n\nEDUCATION\n\nPG-DAC\n\nCDAC ACTS\n\n2017\n\nBachelor of Engg in Information Technology\n\nShivaji University Kolhapur -  Kolhapur, Maharashtra\n\n2016\n\nSKILLS\n\nDatabase (Less than 1 year), HTML (Less than 1 year), Linux. (Less than 1 year), MICROSOFT\nACCESS (Less than 1 year), MICROSOFT WINDOWS (Less than 1 year)\n\nADDITIONAL INFORMATION\n\nTECHNICAL SKILLS:\n\n• 

In [27]:
# loading the dataset
df = pd.read_json('./data.json' , lines = True)
df.head()

,content,annotation,extras,metadata
0,Afreen Jamadar\nActive member of IIIT Committe...,"[{'label': ['Email Address'], 'points': [{'sta...",NaN,"{'first_done_at': 1527844872000, 'last_updated..."
1,Alok Khandai\nOperational Analyst (SQL DBA) En...,"[{'label': ['Skills'], 'points': [{'start': 80...",NaN,"{'first_done_at': 1527845028000, 'last_updated..."
2,Anvitha Rao\nAutomation developer\n\n- Email m...,"[{'label': ['Links'], 'points': [{'start': 288...",NaN,"{'first_done_at': 1527744637000, 'last_updated..."
3,arjun ks\nSenior Program coordinator - oracle ...,"[{'label': ['Skills'], 'points': [{'start': 50...",NaN,"{'first_done_at': 1527834843000, 'last_updated..."
4,"Arun Elumalai\nQA Tester\n\nChennai, Tamil Nad...","[{'label': ['Skills'], 'points': [{'start': 19...",NaN,"{'first_done_at': 1527847268000, 'last_updated..."


In [28]:
# Since, 'extras' column contains no information we can drop the column
df = df.drop(['extras'], axis=1)
df.head()

,content,annotation,metadata
0,Afreen Jamadar\nActive member of IIIT Committe...,"[{'label': ['Email Address'], 'points': [{'sta...","{'first_done_at': 1527844872000, 'last_updated..."
1,Alok Khandai\nOperational Analyst (SQL DBA) En...,"[{'label': ['Skills'], 'points': [{'start': 80...","{'first_done_at': 1527845028000, 'last_updated..."
2,Anvitha Rao\nAutomation developer\n\n- Email m...,"[{'label': ['Links'], 'points': [{'start': 288...","{'first_done_at': 1527744637000, 'last_updated..."
3,arjun ks\nSenior Program coordinator - oracle ...,"[{'label': ['Skills'], 'points': [{'start': 50...","{'first_done_at': 1527834843000, 'last_updated..."
4,"Arun Elumalai\nQA Tester\n\nChennai, Tamil Nad...","[{'label': ['Skills'], 'points': [{'start': 19...","{'first_done_at': 1527847268000, 'last_updated..."


In [29]:
# Since, 'metadata' column contains no information we can drop the column
df = df.drop(['metadata'], axis=1)
df.head()

,content,annotation
0,Afreen Jamadar\nActive member of IIIT Committe...,"[{'label': ['Email Address'], 'points': [{'sta..."
1,Alok Khandai\nOperational Analyst (SQL DBA) En...,"[{'label': ['Skills'], 'points': [{'start': 80..."
2,Anvitha Rao\nAutomation developer\n\n- Email m...,"[{'label': ['Links'], 'points': [{'start': 288..."
3,arjun ks\nSenior Program coordinator - oracle ...,"[{'label': ['Skills'], 'points': [{'start': 50..."
4,"Arun Elumalai\nQA Tester\n\nChennai, Tamil Nad...","[{'label': ['Skills'], 'points': [{'start': 19..."


In [3]:
def mergeIntervals(intervals):
    sorted_by_lower_bound = sorted(intervals, key=lambda tup: tup[0])
    merged = []
    for higher in sorted_by_lower_bound:
        if not merged:
            merged.append(higher)
        else:
            lower = merged[-1]
            if higher[0] <= lower[1]:
                if lower[2] is higher[2]:
                    upper_bound = max(lower[1], higher[1])
                    merged[-1] = (lower[0], upper_bound, lower[2])
                else:
                    if lower[1] > higher[1]:
                        merged[-1] = lower
                    else:
                        merged[-1] = (lower[0], higher[1], higher[2])
            else:
                merged.append(higher)
    return merged

In [31]:
#intervals = [(1, 3), (2, 6), (8, 10), (15, 18), (17, 20)]
# After merging intervals, the expected result should be:
# [(1, 6), (8, 10), (15, 20)]

In [32]:
print ("notre data est un dictionnaire qui contient content et annotation Annotations est une liste qui contient des dictionnaire (chaque dictionnaire est construit par label et points) points est une liste (elle contient 1 seul element) de dictionnaire (start , end , text)")

notre data est un dictionnaire qui contient content et annotation Annotations est une liste qui contient des dictionnaire (chaque dictionnaire est construit par label et points) points est une liste (elle contient 1 seul element) de dictionnaire (start , end , text)


In [33]:
def get_entities(df):
    entities = []
    for i in range(len(df)):
        entity = []
        if df['annotation'][i] is not None: 
            for annot in df['annotation'][i]:
                try:
                    ent = entity_dict[annot['label'][0]]
                    start = annot['points'][0]['start']
                    end = annot['points'][0]['end'] + 1
                    entity.append((start, end, ent))
                except:
                    pass
            entity = mergeIntervals(entity)
            entities.append(entity)
        else :
            entities.append([])
    return entities

In [34]:
    # Adding a new column 'entities'
    df['entities'] = get_entities(df)
    df.head()

,content,annotation,entities
0,Afreen Jamadar\nActive member of IIIT Committe...,"[{'label': ['Email Address'], 'points': [{'sta...","[(0, 14, NAME), (62, 68, LOC), (104, 148, EMAI..."
1,Alok Khandai\nOperational Analyst (SQL DBA) En...,"[{'label': ['Skills'], 'points': [{'start': 80...","[(0, 12, NAME), (13, 51, DESIG), (54, 60, COMP..."
2,Anvitha Rao\nAutomation developer\n\n- Email m...,"[{'label': ['Links'], 'points': [{'start': 288...","[(0, 11, NAME), (12, 32, DESIG), (56, 97, EMAI..."
3,arjun ks\nSenior Program coordinator - oracle ...,"[{'label': ['Skills'], 'points': [{'start': 50...","[(0, 8, NAME), (9, 35, DESIG), (38, 58, COMPAN..."
4,"Arun Elumalai\nQA Tester\n\nChennai, Tamil Nad...","[{'label': ['Skills'], 'points': [{'start': 19...","[(0, 13, NAME), (14, 24, DESIG), (25, 32, LOC)..."


In [35]:
def get_train_data(df):
    tags = []
    sentences = []
    for i in range(len(df)):
        text = df['content'][i]
        entities = df['entities'][i]
        doc = nlp(text)
        tag = offsets_to_biluo_tags(doc, entities)
        tmp = pd.DataFrame([list(doc), tag]).T
        loc = []
        for i in range(len(tmp)):
            if tmp[0][i].text == '.' and tmp[1][i] == 'O':
                loc.append(i)
        loc.append(len(doc))
        last = 0
        data = []
        for pos in loc:
            data.append([list(doc)[last:pos], tag[last:pos]])
            last = pos
        for d in data:
            tag = ['O' if t == '-' else t for t in d[1]]
            if len(set(tag)) > 1:
                sentences.append(d[0])
                tags.append(tag)
    return sentences, tags

In [36]:
#sentences hia liste mta3 sentence metkawwna men (tokens) and each token has correspondig tag
#koll manel9aw point "." na3mlou sentence jdida
sentences, tags = get_train_data(df)
len(sentences), len(tags)

C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Afreen Jamadar
Active member of IIIT Committee in ..." with entities "[(0, 14, 'NAME'), (62, 68, 'LOC'), (104, 148, 'EMA...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Alok Khandai
Operational Analyst (SQL DBA) Enginee..." with entities "[(0, 12, 'NAME'), (13, 51, 'DESIG'), (54, 60, 'COM...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\spacy\training\iob_utils.py:149: UserW

(3108, 3108)

In [37]:
# Print first few sentences and their corresponding tags
for sent, tag in zip(sentences[:2], tags[:2]):
    print("Sentence:", sent)
    print("Tags:", tag)
    print()

Sentence: [Afreen, Jamadar, 
, Active, member, of, IIIT, Committee, in, Third, year, 

, Sangli, ,, Maharashtra, -, Email, me, on, Indeed, :, indeed.com/r/Afreen-Jamadar/8baf379b705e37c6, 

, I, wish, to, use, my, knowledge, ,, skills, and, conceptual, understanding, to, create, excellent, team, 
, environments, and, work, consistently, achieving, organization, objectives, believes, in, taking, initiative, 
, and, work, to, excellence, in, my, work]
Tags: ['B-NAME', 'L-NAME', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'U-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'U-EMAIL', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Sentence: [., 

, WORK, EXPERIENCE, 

, Active, member, of, IIIT, Committee, in, Third, year, 

, Cisco, Networking, -,  , Kanpur, ,, Uttar, Pradesh, 

, organized, by, Techkriti, IIT, Kanpur, and, Azure, Skynet]
Tags: ['O', 'O'

In [38]:
#bech nzidou 3tags 
#'X': This tag often represents tokens that are not part of any named entity or class. In other words, it's a placeholder tag for tokens that don't fit into any predefined category.
#[CLS]: This token is a special symbol added to the beginning of input sequences.
#[SEP]: This token is used to separate segments or sentences in the input sequence. 
tag_vals = set(['X', '[CLS]', '[SEP]'])
for i in range(len(tags)):
    tag_vals = tag_vals.union(tags[i])
tag_vals

{'B-CLG',
 'B-COMPANY',
 'B-DEG',
 'B-DESIG',
 'B-EMAIL',
 'B-GRADYEAR',
 'B-LOC',
 'B-NAME',
 'B-SKILLS',
 'B-YOE',
 'I-CLG',
 'I-COMPANY',
 'I-DEG',
 'I-DESIG',
 'I-EMAIL',
 'I-GRADYEAR',
 'I-LOC',
 'I-NAME',
 'I-SKILLS',
 'I-YOE',
 'L-CLG',
 'L-COMPANY',
 'L-DEG',
 'L-DESIG',
 'L-EMAIL',
 'L-GRADYEAR',
 'L-LOC',
 'L-NAME',
 'L-SKILLS',
 'L-YOE',
 'O',
 'U-CLG',
 'U-COMPANY',
 'U-DEG',
 'U-DESIG',
 'U-EMAIL',
 'U-GRADYEAR',
 'U-LOC',
 'U-NAME',
 'U-SKILLS',
 'U-YOE',
 'X',
 '[CLS]',
 '[SEP]'}

In [39]:
tag2idx = {t: i for i, t in enumerate(tag_vals)}
tag2idx

{'L-LOC': 0,
 'I-EMAIL': 1,
 'B-DEG': 2,
 'L-SKILLS': 3,
 'U-CLG': 4,
 '[CLS]': 5,
 '[SEP]': 6,
 'U-COMPANY': 7,
 'B-GRADYEAR': 8,
 'L-COMPANY': 9,
 'U-NAME': 10,
 'O': 11,
 'U-DEG': 12,
 'L-EMAIL': 13,
 'L-YOE': 14,
 'U-DESIG': 15,
 'B-CLG': 16,
 'U-SKILLS': 17,
 'L-DESIG': 18,
 'B-YOE': 19,
 'I-SKILLS': 20,
 'L-NAME': 21,
 'L-GRADYEAR': 22,
 'I-DEG': 23,
 'U-EMAIL': 24,
 'I-COMPANY': 25,
 'U-GRADYEAR': 26,
 'B-DESIG': 27,
 'U-LOC': 28,
 'I-GRADYEAR': 29,
 'B-COMPANY': 30,
 'I-CLG': 31,
 'L-CLG': 32,
 'B-LOC': 33,
 'I-YOE': 34,
 'I-LOC': 35,
 'B-EMAIL': 36,
 'I-NAME': 37,
 'B-NAME': 38,
 'X': 39,
 'B-SKILLS': 40,
 'L-DEG': 41,
 'I-DESIG': 42,
 'U-YOE': 43}

In [40]:
idx2tag = {tag2idx[key] : key for key in tag2idx.keys()}
idx2tag

{0: 'L-LOC',
 1: 'I-EMAIL',
 2: 'B-DEG',
 3: 'L-SKILLS',
 4: 'U-CLG',
 5: '[CLS]',
 6: '[SEP]',
 7: 'U-COMPANY',
 8: 'B-GRADYEAR',
 9: 'L-COMPANY',
 10: 'U-NAME',
 11: 'O',
 12: 'U-DEG',
 13: 'L-EMAIL',
 14: 'L-YOE',
 15: 'U-DESIG',
 16: 'B-CLG',
 17: 'U-SKILLS',
 18: 'L-DESIG',
 19: 'B-YOE',
 20: 'I-SKILLS',
 21: 'L-NAME',
 22: 'L-GRADYEAR',
 23: 'I-DEG',
 24: 'U-EMAIL',
 25: 'I-COMPANY',
 26: 'U-GRADYEAR',
 27: 'B-DESIG',
 28: 'U-LOC',
 29: 'I-GRADYEAR',
 30: 'B-COMPANY',
 31: 'I-CLG',
 32: 'L-CLG',
 33: 'B-LOC',
 34: 'I-YOE',
 35: 'I-LOC',
 36: 'B-EMAIL',
 37: 'I-NAME',
 38: 'B-NAME',
 39: 'X',
 40: 'B-SKILLS',
 41: 'L-DEG',
 42: 'I-DESIG',
 43: 'U-YOE'}

In [41]:
#This code snippet sets up PyTorch to utilize GPU (CUDA) if it's available, 
#and it also determines the number of available GPUs.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()

In [42]:
tokenizer = BertTokenizer.from_pretrained('bert-base-cased', do_lower_case=False)

In [43]:
def get_tokenized_train_data(sentences, tags):
    tokenized_texts = []
    word_piece_labels = []
    for word_list, label in zip(sentences, tags):
        temp_lable = ['[CLS]']
        temp_token = ['[CLS]']
        for word, lab in zip(word_list, label):
            token_list = tokenizer.tokenize(word.text) 
            for m, token in enumerate(token_list):
                temp_token.append(token)
                if m == 0:
                    temp_lable.append(lab)
                else:
                    temp_lable.append('X') 
        temp_lable.append('[SEP]')
        temp_token.append('[SEP]')
        tokenized_texts.append(temp_token)
        word_piece_labels.append(temp_lable)
    return tokenized_texts, word_piece_labels

In [44]:
#tokenized_texts hia liste de liste koll element fiha liste fih les tokens mta3 koll jomla 
tokenized_texts, word_piece_labels = get_tokenized_train_data(sentences, tags)
len(tokenized_texts[0]),len(tokenized_texts)

(87, 3108)

In [45]:
print(tokenized_texts[0])
print(word_piece_labels[0])

['[CLS]', 'A', '##free', '##n', 'Jam', '##ada', '##r', 'Active', 'member', 'of', 'III', '##T', 'Committee', 'in', 'Third', 'year', 'Sang', '##li', ',', 'Maharashtra', '-', 'Em', '##ail', 'me', 'on', 'Indeed', ':', 'indeed', '.', 'com', '/', 'r', '/', 'A', '##free', '##n', '-', 'Jam', '##ada', '##r', '/', '8', '##ba', '##f', '##37', '##9', '##b', '##70', '##5', '##e', '##37', '##c', '##6', 'I', 'wish', 'to', 'use', 'my', 'knowledge', ',', 'skills', 'and', 'conceptual', 'understanding', 'to', 'create', 'excellent', 'team', 'environments', 'and', 'work', 'consistently', 'achieving', 'organization', 'objectives', 'believes', 'in', 'taking', 'initiative', 'and', 'work', 'to', 'excellence', 'in', 'my', 'work', '[SEP]']
['[CLS]', 'B-NAME', 'X', 'X', 'L-NAME', 'X', 'X', 'O', 'O', 'O', 'O', 'X', 'O', 'O', 'O', 'O', 'U-LOC', 'X', 'O', 'O', 'O', 'O', 'X', 'O', 'O', 'O', 'O', 'U-EMAIL', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X

In [46]:
#matrice mta3 tokens_ids
MAX_LEN = 512 #MAX_LEN = 512: This variable specifies the maximum length of the input sequences. Any sequences longer than this length will be truncated, 
                              #and any sequences shorter than this length will be padded.
bs = 8  #batch size
#Each token in the txt is mapped to its unique integer ID.
input_ids = pad_sequences([tokenizer.convert_tokens_to_ids(txt) for txt in tokenized_texts],
                          maxlen=MAX_LEN, dtype="long", truncating="post", padding="post")
print(len(input_ids[0]))
print(input_ids[0])

Token indices sequence length is longer than the specified maximum  sequence length for this BERT model (679 > 512). Running this sequence through BERT will result in indexing errors
Token indices sequence length is longer than the specified maximum  sequence length for this BERT model (977 > 512). Running this sequence through BERT will result in indexing errors
Token indices sequence length is longer than the specified maximum  sequence length for this BERT model (567 > 512). Running this sequence through BERT will result in indexing errors
Token indices sequence length is longer than the specified maximum  sequence length for this BERT model (1054 > 512). Running this sequence through BERT will result in indexing errors
Token indices sequence length is longer than the specified maximum  sequence length for this BERT model (1231 > 512). Running this sequence through BERT will result in indexing errors
Token indices sequence length is longer than the specified maximum  sequence length

512
[  101   138 26743  1179 13263  7971  1197 17244  1420  1104  2684  1942
  2341  1107  4180  1214 22409  2646   117 12626   118 18653 11922  1143
  1113 10364   131  5750   119  3254   120   187   120   138 26743  1179
   118 13263  7971  1197   120   129  2822  2087 26303  1580  1830 20829
  1571  1162 26303  1665  1545   146  3683  1106  1329  1139  3044   117
  4196  1105 20046  4287  1106  2561  6548  1264 10152  1105  1250 10887
 11190  2369 11350  6616  1107  1781  7191  1105  1250  1106 14509  1107
  1139  1250   102     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     

In [47]:
#matrice mta3 tags mta3 koll sentence (hna les id a7na 3tinahom bel fct tag2idx)
tags = pad_sequences([[tag2idx.get(l) for l in lab] for lab in word_piece_labels], maxlen=MAX_LEN, value=tag2idx["O"], 
                     padding="post", dtype="long", truncating="post")
print(len(tags))
print(len(tags[0]))
print(tags[0])

3108
512
[ 5 38 39 39 21 39 39 11 11 11 11 39 11 11 11 11 28 39 11 11 11 11 39 11
 11 11 11 24 39 39 39 39 39 39 39 39 39 39 39 39 39 39 39 39 39 39 39 39
 39 39 39 39 39 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11  6 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11

In [48]:
#fel matrice mta3 token_ids i4a ken el valeur >0 te5ou 1
attention_masks = [[float(i>0) for i in ii] for ii in input_ids]
print(attention_masks[0])

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [52]:
#Training w validation lel : input , tags , masks
tr_inputs, val_inputs, tr_tags, val_tags, tr_masks, val_masks = train_test_split(input_ids, tags, attention_masks, random_state=2020,test_size=0.3)

In [53]:
#By converting the input data, labels, and masks into PyTorch tensors, we ensure that they are in the appropriate format for consumption by PyTorch-based deep learning models, 
#enabling efficient training, validation, and inference processes.
tr_inputs = torch.tensor(tr_inputs)
val_inputs = torch.tensor(val_inputs)
tr_tags = torch.tensor(tr_tags)
val_tags = torch.tensor(val_tags)
tr_masks = torch.tensor(tr_masks)
val_masks = torch.tensor(val_masks)

In [54]:
train_data = TensorDataset(tr_inputs, tr_masks, tr_tags) #TensorDataset: TensorDataset is a PyTorch utility class that allows you to create a dataset from a set of tensors. In this case, it's used to combine the input data, attention masks, and tags or labels into a single dataset.
train_sampler = RandomSampler(train_data) #shuffles the data before sampling, which helps in introducing randomness and avoiding bias during training.
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=bs) #DataLoader: DataLoader is another PyTorch utility class that provides an iterable over a dataset. It supports various features like batching, shuffling, and parallel data loading. By using DataLoader, we can efficiently iterate over the dataset in batches during training and validation.
valid_data = TensorDataset(val_inputs, val_masks, val_tags)
valid_sampler = SequentialSampler(valid_data) #samples data sequentially without shuffling, which is typically used for validation or testing datasets to ensure reproducible results.
valid_dataloader = DataLoader(valid_data, sampler=valid_sampler, batch_size=bs)

In [55]:
model = BertForTokenClassification.from_pretrained("bert-base-cased", num_labels=len(tag2idx))

In [56]:
import torch
print(torch.cuda.is_available())

True


In [57]:
model.cuda(); #bech twalli te5dem 3al gpu

In [58]:
FULL_FINETUNING = True
if FULL_FINETUNING:
    param_optimizer = list(model.named_parameters()) 
    optimizer_grouped_parameters = [
        {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
         'weight_decay_rate': 0.01},
        {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
         'weight_decay_rate': 0.0}
    ]
else:
    param_optimizer = list(model.classifier.named_parameters()) 
    optimizer_grouped_parameters = [{"params": [p for n, p in param_optimizer]}]
optimizer = Adam(optimizer_grouped_parameters, lr=3e-5)

In [59]:
# Get the first batch
first_batch = next(iter(train_dataloader))
# Print the contents of the first batch
print("First batch contents:")
print("Input IDs:", first_batch[0])
print("Attention Mask:", first_batch[1])
print("Labels:", first_batch[2])
# Check data types and shapes
input_ids, attention_mask, labels = first_batch
print("Input IDs shape:", input_ids.shape)
print("Attention Mask shape:", attention_mask.shape)
print("Labels shape:", labels.shape)
print("Input IDs dtype:", input_ids.dtype)
print("Attention Mask dtype:", attention_mask.dtype)
print("Labels dtype:", labels.dtype)

First batch contents:
Input IDs: tensor([[  101,   119,   142,  ...,     0,     0,     0],
        [  101,   119,  3100,  ...,     0,     0,     0],
        [  101,   119,   794,  ...,     0,     0,     0],
        ...,
        [  101,   119,  9612,  ...,     0,     0,     0],
        [  101, 17784,  2340,  ...,     0,     0,     0],
        [  101,  7085,  6385,  ...,     0,     0,     0]], dtype=torch.int32)
Attention Mask: tensor([[1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.]])
Labels: tensor([[ 5, 11, 11,  ..., 11, 11, 11],
        [ 5, 11, 11,  ..., 11, 11, 11],
        [ 5, 11, 11,  ..., 11, 11, 11],
        ...,
        [ 5, 11, 11,  ..., 11, 11, 11],
        [ 5, 38, 39,  ..., 11, 11, 11],
        [ 5, 38, 39,  ..., 11, 11, 11]], dtype=torch.int32)
Input IDs shape: torch.Size([8, 512

In [60]:
# Iterate over the DataLoader and print batch information
batch_count = 0
for batch in train_dataloader:
    batch_count += 1
    print(f"Processing batch {batch_count}")
# Print the total number of batches
print(f"Total number of batches: {batch_count}")

Processing batch 1
Processing batch 2
Processing batch 3
Processing batch 4
Processing batch 5
Processing batch 6
Processing batch 7
Processing batch 8
Processing batch 9
Processing batch 10
Processing batch 11
Processing batch 12
Processing batch 13
Processing batch 14
Processing batch 15
Processing batch 16
Processing batch 17
Processing batch 18
Processing batch 19
Processing batch 20
Processing batch 21
Processing batch 22
Processing batch 23
Processing batch 24
Processing batch 25
Processing batch 26
Processing batch 27
Processing batch 28
Processing batch 29
Processing batch 30
Processing batch 31
Processing batch 32
Processing batch 33
Processing batch 34
Processing batch 35
Processing batch 36
Processing batch 37
Processing batch 38
Processing batch 39
Processing batch 40
Processing batch 41
Processing batch 42
Processing batch 43
Processing batch 44
Processing batch 45
Processing batch 46
Processing batch 47
Processing batch 48
Processing batch 49
Processing batch 50
Processin

In [61]:
import torch
from tqdm import trange
epochs = 10
max_grad_norm = 1.0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
try:
    for epoch in trange(epochs, desc="Epoch"):
        print(f"Epoch {epoch + 1} started")
        model.train()
        tr_loss = 0
        correct_predictions = 0
        total_predictions = 0
        nb_tr_steps = 0
        for step, batch in enumerate(train_dataloader):
            if step % 10 == 0:
                print(f"Processing batch {step}")
            batch = tuple(t.to(device) for t in batch)
            b_input_ids, b_input_mask, b_labels = batch
            b_labels = b_labels.long()
            outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
            loss = outputs[0] if isinstance(outputs, tuple) else outputs
            logits = outputs[1] if isinstance(outputs, tuple) else None 
            loss.backward()
            tr_loss += loss.item()
            nb_tr_steps += 1
            if logits is not None:
                preds = torch.argmax(logits, dim=1)
                correct_predictions += (preds == b_labels).sum().item()
                total_predictions += b_labels.size(0)
            torch.nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=max_grad_norm)
            optimizer.step()
            model.zero_grad()
        avg_loss = tr_loss / nb_tr_steps
        print(f"📊 Epoch {epoch + 1} - Train Loss: {avg_loss:.4f}")
        if epoch  == 8:
            print("🛑 Reached 5th epoch. Stopping and saving model...")
            torch.save(model.state_dict(), "eight_epoch.pt")
            print("✅ Model saved as 'eight_epoch.pt'")
            break
except KeyboardInterrupt:
    print("🛑 Training manually interrupted. Saving model...")
    torch.save(model.state_dict(), "interruptedd_model.pt")
    print("✅ Model saved as 'interruptedd_model.pt'")

Epoch:   0%|                                                                                    | 0/10 [00:00<?, ?it/s]

Epoch 1 started
Processing batch 0
Processing batch 10
Processing batch 20
Processing batch 30
Processing batch 40
Processing batch 50
Processing batch 60
Processing batch 70
Processing batch 80
Processing batch 90
Processing batch 100
Processing batch 110
Processing batch 120
Processing batch 130
Processing batch 140
Processing batch 150
Processing batch 160
Processing batch 170
Processing batch 180
Processing batch 190
Processing batch 200
Processing batch 210
Processing batch 220
Processing batch 230
Processing batch 240
Processing batch 250
Processing batch 260
Processing batch 270


Epoch:  10%|██████▉                                                              | 1/10 [1:21:44<12:15:39, 4904.34s/it]

📊 Epoch 1 - Train Loss: 0.5466
Epoch 2 started
Processing batch 0
Processing batch 10
Processing batch 20
Processing batch 30
Processing batch 40
Processing batch 50
Processing batch 60
Processing batch 70
Processing batch 80
Processing batch 90
Processing batch 100
Processing batch 110
Processing batch 120
Processing batch 130
Processing batch 140
Processing batch 150
Processing batch 160
Processing batch 170
Processing batch 180
Processing batch 190
Processing batch 200
Processing batch 210
Processing batch 220
Processing batch 230
Processing batch 240
Processing batch 250
Processing batch 260
Processing batch 270


Epoch:  20%|█████████████▊                                                       | 2/10 [2:42:30<10:49:20, 4870.01s/it]

...

📊 Epoch 7 - Train Loss: 0.0722
Epoch 8 started
Processing batch 0
Processing batch 10
Processing batch 20
Processing batch 30
Processing batch 40
Processing batch 50
Processing batch 60
Processing batch 70
Processing batch 80
Processing batch 90
Processing batch 100
Processing batch 110
Processing batch 120
Processing batch 130
Processing batch 140
Processing batch 150
Processing batch 160
Processing batch 170
Processing batch 180
Processing batch 190
Processing batch 200
Processing batch 210
Processing batch 220
Processing batch 230
Processing batch 240
Processing batch 250
Processing batch 260
Processing batch 270


Epoch:  80%|███████████████████████████████████████████████████████▏             | 8/10 [10:47:14<2:41:21, 4840.60s/it]

📊 Epoch 8 - Train Loss: 0.0573
Epoch 9 started
Processing batch 0
Processing batch 10
Processing batch 20
Processing batch 30
Processing batch 40
Processing batch 50
Processing batch 60
Processing batch 70
Processing batch 80
Processing batch 90
Processing batch 100
Processing batch 110
Processing batch 120
Processing batch 130
Processing batch 140
Processing batch 150
Processing batch 160
Processing batch 170
Processing batch 180
Processing batch 190
Processing batch 200
Processing batch 210
Processing batch 220
Processing batch 230
Processing batch 240
Processing batch 250
Processing batch 260
Processing batch 270
📊 Epoch 9 - Train Loss: 0.0477
🛑 Reached 5th epoch. Stopping and saving model...


Epoch:  80%|███████████████████████████████████████████████████████▏             | 8/10 [12:07:15<3:01:48, 5454.46s/it]

✅ Model saved as 'eight_epoch.pt'
